## 🌟 STEP 0 - Load & format the dataset and save the "abstracts_x(i.e no. of rows).csv" file

In [ ]:
#   !pip install datasets

In [ ]:
from datasets import load_dataset

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

NUM_PAPERS = 5000
SEED = 42

output_ABSTRACTS_csv = f"abstracts_{NUM_PAPERS}.csv"
input_ABSTRACTS_csv = f"abstracts_{NUM_PAPERS}.csv"

output_ANNOTATIONS_json = f"auto_annotations_{NUM_PAPERS}.json"
# input_ANNOTATIONS_json = f"auto_annotations_{NUM_PAPERS}.json"

# output_GOLD_json = f"gold_annotations_{NUM_PAPERS}.json"
# input_GOLD_json = f"gold_annotations_{NUM_PAPERS}.json"

# output_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"
# input_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"

# output_BIO_json = f"bio_dataset_{NUM_PAPERS}.json"
# input_BIO_json = f"bio_dataset_{NUM_PAPERS}.json"

API_KEY = "Your API"

BATCH_SIZE = 2

START_FROM = 0
# Allows the annotation process to resume from the last processed abstract (index) if it gets interrupted

SAVE_EVERY = 10
# Saves the annotations to a JSON file after every N abstracts to minimize data loss

SLEEP_TIME = 6
# Waits x seconds between Gemini API requests to avoid rate limit errors

ERROR_SLEEP_TIME = 20

MODEL_NAME = "gemini-3.1-flash-lite"

In [ ]:
print(f"Total Papers : {NUM_PAPERS}\n")
print(f"Output abstracts file : {output_ABSTRACTS_csv}")
print(f"Input abstracts file : {input_ABSTRACTS_csv}\n")
print(f"Output annotations file : {output_ANNOTATIONS_json}")
print(f"Input annotations file : {input_ANNOTATIONS_json}\n")
print(f"Output gold file : {output_GOLD_json}")
print(f"Input gold file : {input_GOLD_json}\n")
print(f"Output normalized gold file : {output_NORMALIZED_GOLD_json}")
print(f"Input normalized gold file : {input_NORMALIZED_GOLD_json}\n")
print(f"Output bio file : {output_BIO_json}")
print(f"Input bio file : {input_BIO_json}")

Total Papers : 1430

Output abstracts file : abstracts_1430.csv
Input abstracts file : abstracts_1430.csv

Output annotations file : auto_annotations_1430.json
Input annotations file : auto_annotations_1430.json

Output gold file : gold_annotations_1430.json
Input gold file : gold_annotations_1430.json

Output normalized gold file : normalized_gold_annotations_1430.json
Input normalized gold file : normalized_gold_annotations_1430.json

Output bio file : bio_dataset_1430.json
Input bio file : bio_dataset_1430.json


In [ ]:
# LOAD DATASET
dataset = load_dataset("CShorten/ML-ArXiv-Papers" , split = 'train')
print(f"Total Papers : {len(dataset)}")

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Total Papers : 117592


In [ ]:
print(dataset)

NameError: name 'dataset' is not defined

In [ ]:
dataset[50]

{'Unnamed: 0.1': 50,
 'Unnamed: 0': 50.0,
 'title': 'Consistency of trace norm minimization',
 'abstract': '  Regularization by the sum of singular values, also referred to as the trace\nnorm, is a popular technique for estimating low rank rectangular matrices. In\nthis paper, we extend some of the consistency results of the Lasso to provide\nnecessary and sufficient conditions for rank consistency of trace norm\nminimization with the square loss. We also provide an adaptive version that is\nrank consistent even when the necessary condition for the non adaptive version\nis not fulfilled.\n'}

In [ ]:
# Shuffle dataset
dataset = dataset.shuffle(seed = SEED)

In [ ]:
# Select required number of papers
dataset_subset = dataset.select(range(NUM_PAPERS))

In [ ]:
print(f"Selected Papers : {len(dataset_subset)}")

Selected Papers : 5000


In [ ]:
import pandas as pd

In [ ]:
# Convert to DataFrame
df = dataset_subset.to_pandas()

In [ ]:
# Remove unwanted columns
df = df.drop(columns = ["Unnamed: 0", "Unnamed: 0.1"])

In [ ]:
# Remove leading/trailing spaces
df["title"] = df["title"].str.strip()
df["abstract"] = df["abstract"].str.strip()

# Replace multiple whitespaces/newlines/tabs with single space
df["title"] = df["title"].str.replace(r"\s+", " ", regex=True)
df["abstract"] = df["abstract"].str.replace(r"\s+", " ", regex=True)

In [ ]:
df.head()

NameError: name 'df' is not defined

In [ ]:
# Save the abstract CSV file
df.to_csv(
  output_ABSTRACTS_csv,
  index=True,
  index_label="id"
)

print(f"{output_ABSTRACTS_csv} saved successfully!")

abstracts_1430.csv saved successfully!



### Pipeline

```
STEP 1  Install Libraries
STEP 2  Imports
STEP 3  Gemini API
STEP 4  Configuration (Sirf yahan filename change karna)
STEP 5  Load CSV
STEP 6  Auto Annotation
STEP 7  Save JSON
STEP 8  Clean Dataset (Bad entities + Canonical Mapping)
STEP 9  Manual Review (Optional)
STEP 10 Gold Dataset Save
STEP 11 BIO Conversion
STEP 12 Train SciBERT
STEP 13 Evaluation
STEP 14 Save Model
STEP 15 Inference
```


## 🌟 STEP 1 - Perform the annotation and save the "auto_annotations_x(i.e no. of rows).json"

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

input_ABSTRACTS_csv = "abstracts_5000.csv"
output_ANNOTATIONS_json = f"auto_annotations_{NUM_PAPERS}.json"

MODEL_NAME = "gemini-3.1-flash-lite"

### 1.1. Install Libraries

In [ ]:
# !pip -q install -U google-genai
# !pip -q install datasets
# !pip -q install transformers
# !pip -q install seqeval
# !pip -q install accelerate

### 1.2. Import Required Libraries

In [ ]:
import os
import json
import time
import pandas as pd

from google import genai
from google.genai import types

from tqdm import tqdm

### 1.3. Gemini API

In [ ]:
# API_KEY = "Your  API"

client = genai.Client(api_key = API_KEY)

print("Gemini Connected Successfully!")

Gemini Connected Successfully!


### → Change values here

In [ ]:
BATCH_SIZE = 2

START_FROM = 0
# Allows the annotation process to resume from the last processed abstract (index) if it gets interrupted

SAVE_EVERY = 10
# Saves the annotations to a JSON file after every N abstracts to minimize data loss

SLEEP_TIME = 6
# Waits x seconds between Gemini API requests to avoid rate limit errors

ERROR_SLEEP_TIME = 20

print("Input File :", input_ABSTRACTS_csv)
print("Output File:", output_ANNOTATIONS_json)
print("Gemini Model:", MODEL_NAME)

Input File : abstracts_1430.csv
Output File: auto_annotations_1430.json
Gemini Model: gemini-2.5-flash-lite


### 1.4. Read the abstract (input) csv file

In [ ]:
df = pd.read_csv(input_ABSTRACTS_csv)

In [ ]:
df = df.dropna(subset = ["abstract"]).reset_index(drop = True)
# Removes rows with missing abstracts and resets the DataFrame index for clean sequential processing.

In [ ]:
print(f"Total Abstracts : {len(df)}")

Total Abstracts : 5000


In [ ]:
df.head()

,id,title,abstract
0,0,Epsilon Consistent Mixup: Structural Regulariz...,In this paper we propose $\epsilon$-Consistent...
1,1,Particle Graph Autoencoders and Differentiable...,Autoencoders have useful applications in high ...
2,2,Probabilistic DAG Search,Exciting contemporary machine learning problem...
3,3,Automatic Detection and Classification of Tick...,"Around the globe, ticks are the culprit of tra..."
4,4,Mixing Real and Synthetic Data to Enhance Neur...,Deep neural networks have gained tremendous im...


### 1.5. Define the "create_prompt()" function

In [ ]:
def create_prompt(text):

    return f"""
You are an expert scientific Named Entity Recognition (NER) annotator.

Your task is to identify and extract named entities from research paper abstracts.

Extract entities ONLY for the following four labels:

=========================================
ALGORITHM
=========================================

Definition:
Machine learning algorithms, deep learning models,
neural network architectures, statistical methods,
optimization algorithms, dimensionality reduction methods,
computational frameworks, foundation models,
large language models (LLMs), graph neural networks,
vision models, and diffusion models.

Examples:

Random Forest
Decision Tree
SVM
Support Vector Machine
XGBoost
LightGBM
CatBoost
CNN
Convolutional Neural Network
RNN
Recurrent Neural Network
LSTM
GRU
Transformer
BERT
RoBERTa
GPT
ResNet
VGG
AlexNet
U-Net
DeepONet
GAN
Autoencoder
PCA
SVD
POD
K-means
DBSCAN
Adam
SGD
AdaGrad
RMSProp


Do NOT label generic terms:

model
models
framework
frameworks
approach
approaches
method
methods
technique
techniques
system
systems
architecture
architectures
pipeline
strategy
procedure
mechanism
algorithm
algorithms
surrogate
emulation
reinforcement
sampling efficiency


=========================================
DATASET
=========================================

Definition:
Named datasets, benchmarks, corpora,
collections of samples, and evaluation datasets.

Examples:

ImageNet
CIFAR-10
CIFAR-100
MNIST
Fashion-MNIST
COCO
SQuAD
LibriSpeech
ASVspoof
KITTI
Cityscapes
Pascal VOC
UCI datasets
WikiText
MS MARCO
GLUE
SuperGLUE
Common Crawl
MIMIC-III
MIMIC-IV


Extract only explicitly named datasets.

Do NOT label generic dataset words:

dataset
datasets
training dataset
test dataset
benchmark
benchmarks
corpus
corpora
samples
training data
testing data


=========================================
TASK
=========================================

Definition:
The computational, machine learning,
or analytical problem being solved.

Examples:

Image Classification
Text Classification
Named Entity Recognition
Object Detection
Semantic Segmentation
Instance Segmentation
Speech Recognition
Speaker Verification
Language Identification
Machine Translation
Regression
Prediction
Feature Selection
Sentiment Analysis
Anomaly Detection
Clustering
Recommendation
Forecasting
Information Retrieval
Question Answering
Text Generation
Image Captioning
Machine Reading Comprehension
Knowledge Graph Completion
Link Prediction
Node Classification
Time Series Forecasting
Speech Enhancement
Face Recognition
Pose Estimation


Do NOT classify scientific domains,
application areas,
or research fields as TASK.

Examples NOT TASK:

healthcare
medicine
bioinformatics
computer vision
natural language processing
medical imaging
chemistry
combustion chemistry
physics
biology
finance
cybersecurity


Extract complete task names whenever possible.

Correct:
Image Classification

Incorrect:
Classification


Correct:
Object Detection

Incorrect:
Detection


=========================================
METRIC
=========================================

Definition:
Evaluation metrics used to measure model performance.

Examples:

Accuracy
Precision
Recall
F1-score
F1 score
BLEU
ROUGE
ROC-AUC
AUC
RMSE
MSE
MAE
MAPE
Specificity
Sensitivity
IoU
Dice coefficient
Mean Average Precision
WER
CER
PSNR
SSIM
NDCG
Hit Rate
Top-1 Accuracy
Top-5 Accuracy
Perplexity

Do NOT extract numerical values.

Correct:
Accuracy

Incorrect:
95% Accuracy

Incorrect:
0.92 F1


=========================================
STRICT RULES
=========================================

1. Extract ONLY explicitly mentioned entities from the abstract.

2. Do NOT infer or hallucinate entities.

3. Preserve the original entity text and capitalization.

4. Remove duplicate entity-label pairs. The same entity should appear only once within an abstract.

5. If multiple versions exist, return the longest complete entity.

Example:

Return:
Convolutional Neural Network

Do not return:
Neural Network


6. Every entity must belong to exactly one label.

7. Do NOT label generic words.

8. Do NOT label scientific domains as TASK.

9. Do NOT create entities that are not present in the abstract.

10. Do NOT return LaTeX notation, mathematical expressions,
or escape sequences inside entity names.

11. Convert mathematical notation into plain text whenever possible.

Examples:

Correct:
epsilon-Consistent Mixup

Incorrect:
LaTeX formatted entity names

Correct:
alpha

Incorrect:
Mathematical escape sequences

12. Return entity names as plain UTF-8 text only.

13. The output must be valid JSON and directly parseable using Python json.loads().

14. If no entities exist, return:

{{
    "entities":[]
}}

15. Return ONLY raw JSON.

16. Do NOT use markdown.

17. Do NOT use ```json blocks.

18. Do NOT provide explanations.

19. If a longer entity completely contains a shorter entity, return only the longer entity unless both are explicitly used as separate independent concepts in the abstract.

20. Do not return spelling variants of the same entity. Return only one canonical surface form exactly as it appears in the abstract.
=========================================
OUTPUT FORMAT
=========================================

{{
    "entities":[
        {{
            "entity":"entity text",
            "label":"ALGORITHM"
        }},
        {{
            "entity":"entity text",
            "label":"DATASET"
        }},
        {{
            "entity":"entity text",
            "label":"TASK"
        }},
        {{
            "entity":"entity text",
            "label":"METRIC"
        }}
    ]
}}


=========================================
ABSTRACT
=========================================

{text}
"""

In [ ]:
print(create_prompt("test abstract"))


You are an expert scientific Named Entity Recognition (NER) annotator.

Your task is to identify and extract named entities from research paper abstracts.

Extract entities ONLY for the following four labels:

ALGORITHM

Definition:
Machine learning algorithms, deep learning models,
neural network architectures, statistical methods,
optimization algorithms, dimensionality reduction methods,
computational frameworks, foundation models,
large language models (LLMs), graph neural networks,
vision models, and diffusion models.

Examples:

Random Forest
Decision Tree
SVM
Support Vector Machine
XGBoost
LightGBM
CatBoost
CNN
Convolutional Neural Network
RNN
Recurrent Neural Network
LSTM
GRU
Transformer
BERT
RoBERTa
GPT
ResNet
VGG
AlexNet
U-Net
DeepONet
GAN
Autoencoder
PCA
SVD
POD
K-means
DBSCAN
Adam
SGD
AdaGrad
RMSProp


Do NOT label generic terms:

model
models
framework
frameworks
approach
approaches
method
methods
technique
techniques
system
systems
architecture
architectures
pipeline
str

#### 1.5.1. Define the "create_batch_prompt()" function

In [ ]:
def create_batch_prompt(batch):

    abstracts = ""

    for i, paper in enumerate(batch, start=1):

        abstracts += f"""

=========================================
ABSTRACT {i}
=========================================

{paper["abstract"]}

"""

    batch_instruction = f"""

IMPORTANT

The text above contains {len(batch)} COMPLETELY INDEPENDENT research paper abstracts.

Treat each abstract independently.

Never transfer entities from one abstract to another.

Do NOT merge entities from different abstracts.

An entity appearing in ABSTRACT 1 must NOT appear in the output of ABSTRACT 2 unless it is explicitly present in ABSTRACT 2.

Process each abstract from scratch.

Return ONLY a JSON ARRAY.

Every JSON object must contain exactly one key named "entities".

Do not add any other keys such as "abstract", "id", "title", "index", "confidence", "reasoning", or "notes".

The response MUST begin with '[' and end with ']'.

Do not include any text before or after the JSON array.

The array MUST contain exactly {len(batch)} objects.

Object i corresponds to ABSTRACT i.

The order of JSON objects MUST exactly match the order of the input abstracts.

Each object MUST follow exactly this structure:

{{
    "entities":[
        {{
            "entity":"entity text",
            "label":"ALGORITHM"
        }}
    ]
}}

If an abstract contains no entities, return:

{{
    "entities":[]
}}

Return ONLY the JSON array.

Do NOT use markdown.

Do NOT provide explanations.

Do NOT omit any abstract.

The output must be directly parseable using Python json.loads().

Return EXACTLY one JSON object for every abstract.

If there are 2 abstracts,
return exactly 2 objects.

If there are 5 abstracts,
return exactly 5 objects.

Never return fewer objects.

Never return more objects.

Within each abstract, remove duplicate entities before returning the JSON.

Each abstract is an independent annotation task.

Ignore all previous and following abstracts while extracting entities.
"""

    return create_prompt(abstracts + batch_instruction)

In [ ]:
batch = df.head(2).to_dict("records")

prompt = create_batch_prompt(batch)
print(prompt)
print("\nLength of Prompt :",len(prompt))


You are an expert scientific Named Entity Recognition (NER) annotator.

Your task is to identify and extract named entities from research paper abstracts.

Extract entities ONLY for the following four labels:

ALGORITHM

Definition:
Machine learning algorithms, deep learning models,
neural network architectures, statistical methods,
optimization algorithms, dimensionality reduction methods,
computational frameworks, foundation models,
large language models (LLMs), graph neural networks,
vision models, and diffusion models.

Examples:

Random Forest
Decision Tree
SVM
Support Vector Machine
XGBoost
LightGBM
CatBoost
CNN
Convolutional Neural Network
RNN
Recurrent Neural Network
LSTM
GRU
Transformer
BERT
RoBERTa
GPT
ResNet
VGG
AlexNet
U-Net
DeepONet
GAN
Autoencoder
PCA
SVD
POD
K-means
DBSCAN
Adam
SGD
AdaGrad
RMSProp


Do NOT label generic terms:

model
models
framework
frameworks
approach
approaches
method
methods
technique
techniques
system
systems
architecture
architectures
pipeline
str

### 1.6. Define the "annotate_batch()" function

In [ ]:
def annotate_batch(batch):

    prompt = create_batch_prompt(batch)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace("```json", "").replace("```", "").strip()

    return json.loads(text)

#### 1.6.1. Test on first 20 papers

In [ ]:
test_df = df.head(20).copy()
list_of_dict_papers_20 = test_df.to_dict("records")

print("Total Papers:", len(list_of_dict_papers_20))

In [ ]:
annotations = []

for i in range(0, len(list_of_dict_papers_20), BATCH_SIZE):

    batch = list_of_dict_papers_20[i:i + BATCH_SIZE]

    print(f"Processing Paper No. {i + 1} - {i + len(batch)}")

    try:

        result = annotate_batch(batch)

        for paper, ann in zip(batch, result):

            annotations.append({
                "id": paper["id"],
                "title": paper["title"],
                "annotation": json.dumps(ann, ensure_ascii=False)
            })

        time.sleep(3)

    except Exception as e:

        print(e)
        break

print("\nDone")
print(f"Annotated {len(annotations)} papers")

Processing 1 - 2
Processing 3 - 4
Processing 5 - 6
Processing 7 - 8
Processing 9 - 10
Processing 11 - 12
Processing 13 - 14
Processing 15 - 16
Processing 17 - 18
Processing 19 - 20

Done
Annotated: 20


In [ ]:
OUTPUT_test_JSON = "test_auto_annotations_20.json"

with open(OUTPUT_test_JSON, "w", encoding="utf-8") as f:
    json.dump(annotations, f, indent=2, ensure_ascii=False)

print("Saved:", OUTPUT_test_JSON)

Saved: auto_annotations_20.json


In [ ]:
for item in annotations[:5]:
    print("ID:", item["id"])
    print("Title:", item["title"])
    print(json.loads(item["annotation"]))
    print("\n")

ID: 0
Title: Epsilon Consistent Mixup: Structural Regularization with an Adaptive Consistency-Interpolation Tradeoff
{'entities': [{'entity': 'epsilon-Consistent Mixup', 'label': 'ALGORITHM'}, {'entity': 'Mixup', 'label': 'ALGORITHM'}, {'entity': 'SVHN', 'label': 'DATASET'}, {'entity': 'CIFAR10', 'label': 'DATASET'}, {'entity': 'semi-supervised classification', 'label': 'TASK'}, {'entity': 'accuracy', 'label': 'METRIC'}]}


ID: 1
Title: Particle Graph Autoencoders and Differentiable, Learned Energy Mover's Distance
{'entities': [{'entity': 'Autoencoders', 'label': 'ALGORITHM'}, {'entity': 'graph-based autoencoders', 'label': 'ALGORITHM'}, {'entity': 'graph neural network', 'label': 'ALGORITHM'}, {'entity': 'CERN Large Hadron Collider', 'label': 'DATASET'}, {'entity': 'anomaly detection', 'label': 'TASK'}]}


ID: 2
Title: Probabilistic DAG Search
{'entities': [{'entity': 'Gaussian models', 'label': 'ALGORITHM'}, {'entity': 'Tic-Tac-Toe', 'label': 'DATASET'}, {'entity': 'feature selectio

### 1.7. Annote the "abstracts_x(i.e no. of rows).csv" file

In [ ]:
list_of_dict_papers = df.head(NUM_PAPERS).to_dict("records")

In [ ]:
if os.path.exists(output_ANNOTATIONS_json):

    with open(output_ANNOTATIONS_json, "r", encoding="utf-8") as f:
        annotations = json.load(f)

    START_FROM = len(annotations)

    print(f"Resuming from Paper {START_FROM}")

else:

    annotations = []
    START_FROM = 0

    print("Starting Fresh")

Resuming from Paper 2428


In [ ]:
for i in range(START_FROM, len(list_of_dict_papers), BATCH_SIZE):

    batch = list_of_dict_papers[i:i+BATCH_SIZE]

    print(f"\nProcessing Papers {i+1} - {i+len(batch)}")


    try:

        result = annotate_batch(batch)


        for paper, ann in zip(batch, result):

            annotations.append({

                "id": paper["id"],

                "title": paper["title"],

                "abstract": paper["abstract"],

                "annotation": json.dumps(
                    ann,
                    ensure_ascii=False
                )

            })


        # Save periodically
        if len(annotations) % SAVE_EVERY == 0:


            with open(output_ANNOTATIONS_json, "w", encoding="utf-8") as f:

                json.dump(
                    annotations,
                    f,
                    indent=2,
                    ensure_ascii=False
                )


            print("Progress Saved")


        time.sleep(SLEEP_TIME)



    except Exception as e:


        print("Error:", e)


        with open(output_ANNOTATIONS_json, "w", encoding="utf-8") as f:

            json.dump(
                annotations,
                f,
                indent=2,
                ensure_ascii=False
            )


        print("Emergency Save Done")


        time.sleep(ERROR_SLEEP_TIME)

        break


Processing Papers 2429 - 2430
Progress Saved

Processing Papers 2431 - 2432

Processing Papers 2433 - 2434

Processing Papers 2435 - 2436

Processing Papers 2437 - 2438

Processing Papers 2439 - 2440
Progress Saved

Processing Papers 2441 - 2442

Processing Papers 2443 - 2444

Processing Papers 2445 - 2446

Processing Papers 2447 - 2448

Processing Papers 2449 - 2450
Progress Saved

Processing Papers 2451 - 2452

Processing Papers 2453 - 2454

Processing Papers 2455 - 2456

Processing Papers 2457 - 2458

Processing Papers 2459 - 2460
Progress Saved

Processing Papers 2461 - 2462

Processing Papers 2463 - 2464

Processing Papers 2465 - 2466

Processing Papers 2467 - 2468

Processing Papers 2469 - 2470
Progress Saved

Processing Papers 2471 - 2472

Processing Papers 2473 - 2474

Processing Papers 2475 - 2476

Processing Papers 2477 - 2478

Processing Papers 2479 - 2480
Progress Saved

Processing Papers 2481 - 2482

Processing Papers 2483 - 2484

Processing Papers 2485 - 2486

Processing

In [ ]:
with open(output_ANNOTATIONS_json, "r", encoding="utf-8") as f:

    annotations = json.load(f)


print("Output File :", output_ANNOTATIONS_json)

print("Processed Papers :", len(annotations))

if len(annotations) > 0:

    print("Last Dataset ID :", annotations[-1]["id"])

print("Remaining Papers :", len(list_of_dict_papers)-len(annotations))

Output File : auto_annotations_5000.json
Processed Papers : 3426
Last Dataset ID : 3425
Remaining Papers : 1574


#### 1.7.0. Rename the file "auto_annotations_Y(i.e no. of rows or paper actually annoted).json"

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================
actual_papers = len(annotations)
NUM_PAPERS = actual_papers

print("Using dataset size:", NUM_PAPERS)

Using dataset size: 3426


In [ ]:
# ==========================
# CONFIGURATION
# ==========================
import os

old_file = "auto_annotations_5000.json"
new_file = f"auto_annotations_{actual_papers}.json"

if os.path.exists(old_file):
    os.rename(old_file, new_file)

In [ ]:
# ==========================
# CONFIGURATION
# ==========================
output_ANNOTATIONS_json = f"auto_annotations_{NUM_PAPERS}.json"
input_ANNOTATIONS_json = f"auto_annotations_{NUM_PAPERS}.json"

output_GOLD_json = f"gold_annotations_{NUM_PAPERS}.json"
input_GOLD_json = f"gold_annotations_{NUM_PAPERS}.json"

output_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"
input_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"

output_BIO_json = f"bio_dataset_{NUM_PAPERS}.json"
input_BIO_json = f"bio_dataset_{NUM_PAPERS}.json"

#### 1.7.1. Count the NER label (e.g., TASK, DATASET) & Entities

In [ ]:
# Counts the total occurrences of each NER label across all annotated papers.
# Shows the distribution of entity types in the dataset.

from collections import Counter
counter = Counter()

for item in annotations:

    ents = json.loads(
        item["annotation"]
    )

    for e in ents["entities"]:

        counter[e["label"]] += 1

print(counter)

Counter({'ALGORITHM': 8640, 'TASK': 3609, 'METRIC': 1129, 'DATASET': 1119})


In [ ]:
# Counts the frequency of each entity for every NER label (e.g., TASK, DATASET, MODEL).
# Helps analyze which entities are most common in the annotated dataset.

from collections import defaultdict, Counter

stats = defaultdict(Counter)

for item in annotations:
    ents = json.loads(item["annotation"])

    for e in ents["entities"]:
        stats[e["label"]][e["entity"].strip()] += 1


for label in stats:
    print("\n" + "="*60)
    print(label)
    print("="*60)

    for ent, cnt in stats[label].most_common(50):
        print(f"{cnt:3d}  {ent}")


ALGORITHM
104  deep neural networks
 80  CNN
 73  neural network
 71  neural networks
 57  deep neural network
 54  deep learning
 51  reinforcement learning
 45  convolutional neural network
 45  convolutional neural networks
 42  LSTM
 37  stochastic gradient descent
 36  GAN
 35  CNNs
 34  deep reinforcement learning
 33  DNN
 32  Deep neural networks
 31  SVM
 31  GANs
 29  Transformer
 26  logistic regression
 26  GNNs
 25  gradient descent
 25  SGD
 24  RNN
 24  Deep Neural Networks
 24  autoencoder
 23  recurrent neural networks
 22  BERT
 21  Deep Learning
 21  Gaussian processes
 21  RNNs
 19  Generative Adversarial Networks
 19  Gaussian process
 18  graph neural network
 18  Convolutional Neural Networks
 18  Bayesian optimization
 18  generative adversarial networks
 18  Graph Neural Networks
 17  GNN
 17  VAE
 16  DNNs
 16  ResNet
 16  transfer learning
 15  Support Vector Machine
 15  Convolutional Neural Network
 15  Random Forest
 15  graph neural networks
 14  random 

## 🌟 STEP 2 - Clean the "auto_annotations_x(i.e no. of rows).json" and save the "gold_annotations_x(i.e no. of rows).json"

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

# INPUT FILES
input_ANNOTATIONS_json = f"auto_annotation_{NUM_PAPERS}.json"

# OUTPUT FILES
OUTPUT_gold_JSON = f"gold_annotations_{NUM_PAPERS}.json"

### 2.1. Import Libraries &  Load the "auto_annotations_x(i.e no. of rows).json" file

In [ ]:
import json
from tqdm import tqdm

In [ ]:
with open(input_ANNOTATIONS_json, "r", encoding="utf-8") as f:
    annotations = json.load(f)

print("Loaded:", len(annotations), "papers")

Loaded: 3426 papers


### 2.2. Bad entity

In [ ]:
bad_entities = {

    # Generic concepts
    "training",
    "prediction",
    "generalisation",
    "learning",
    "occupation",
    "reinforcement",
    "sampling efficiency",
    "surrogate",
    "emulation",
    "optimization",
    "qubo",

    # Generic paradigms
    "machine learning",
    "deep learning",
    "artificial intelligence",

    # Generic neural-network umbrella terms
    "neural network",
    "neural networks",
    "deep neural network",
    "deep neural networks",

    # Generic tasks
    "predicting",
    "classify",
    "classification",
    "clustering",
    "regression",
    "linear regression",
    "detection",
    "segmentation",
    "planning",
    "ranking",
    "reconstruction",
    "inference",
    "prediction",

    # Generic learning types
    "online learning",
    "representation learning",
    "supervised learning",
    "unsupervised learning",
    "active learning"
}

In [ ]:
manual_bad_entities = {

    # Wrong dataset predictions
    "cern large hadron collider",
    "tic-tac-toe",

    # Generic dataset words
    "dataset",
    "datasets",
    "benchmark",
    "benchmarks",

    # Generic entity words
    "method",
    "methods",
    "approach",
    "approaches",
    "framework",
    "frameworks",
    "model",
    "models",

    # Noisy datasets from inspection
    "ieee",
    "rooms task",
    "high variance rooms task"
}

### 2.3. Canonical Mapping

In [ ]:
canonical = {

    # CNN
    "cnn": "Convolutional Neural Network",
    "cnns": "Convolutional Neural Network",
    "convolutional neural network": "Convolutional Neural Network",
    "convolutional neural networks": "Convolutional Neural Network",

    # RNN
    "rnn": "Recurrent Neural Network",
    "rnns": "Recurrent Neural Network",
    "recurrent neural network": "Recurrent Neural Network",
    "recurrent neural networks": "Recurrent Neural Network",

    # DNN
    "dnn": "Deep Neural Network",
    "dnns": "Deep Neural Network",

    # GNN
    "gnn": "Graph Neural Network",
    "gnns": "Graph Neural Network",
    "graph neural network": "Graph Neural Network",
    "graph neural networks": "Graph Neural Network",

    # GAN
    "gan": "Generative Adversarial Network",
    "gans": "Generative Adversarial Network",
    "generative adversarial network": "Generative Adversarial Network",
    "generative adversarial networks": "Generative Adversarial Network",

    # Transformer
    "transformer": "Transformer",
    "transformers": "Transformer",

    # SGD
    "sgd": "Stochastic Gradient Descent",
    "stochastic gradient descent": "Stochastic Gradient Descent",

    # LSTM
    "lstm": "Long Short-Term Memory",
    "long short-term memory": "Long Short-Term Memory",

    # GRU
    "gru": "Gated Recurrent Unit",
    "gated recurrent unit": "Gated Recurrent Unit",

    # MLP
    "mlp": "Multi-Layer Perceptron",
    "multi-layer perceptron": "Multi-Layer Perceptron",

    # GCN
    "graph convolutional network": "Graph Convolutional Network",
    "graph convolutional networks": "Graph Convolutional Network",

    # BERT
    "bert-base": "BERT",
    "bert-large": "BERT",

    # Datasets
    "imagenet": "ImageNet",
    "imagenet-subset": "ImageNet-Subset",

    "mnist": "MNIST",

    "cifar10": "CIFAR-10",
    "cifar-10": "CIFAR-10",

    "cifar100": "CIFAR-100",
    "cifar-100": "CIFAR-100",

    "fashionmnist": "Fashion-MNIST",
    "fashion-mnist": "Fashion-MNIST",

    "svhn": "SVHN",

    "coco": "COCO",
    "mscoco": "COCO",

    # Architecture normalization
    "dense net 121": "DenseNet121",
    "densenet 121": "DenseNet121",

    "resnet 34": "ResNet34",
    "resnet 50": "ResNet50",

    "vgg 19": "VGG19",

    # Metrics
    "accuracy": "Accuracy",
    "accuracies": "Accuracy",
    "classification accuracy": "Accuracy",

    "precision": "Precision",
    "recall": "Recall",

    "f1": "F1",
    "f1 score": "F1",
    "f1 scores": "F1",
    "f1-score": "F1",

    "mean squared error": "MSE",
    "mean absolute error": "MAE",

    "word error rate": "WER",
    "word error rates": "WER",

    "auroc": "ROC-AUC",
    "roc-auc": "ROC-AUC"
}

In [ ]:
canonical.update({

    # Metrics
    "f1-scores": "F1",
    "f-measure": "F1",
    "sensitivity": "Sensitivity",

    "classification accuracies": "Accuracy",
    "prediction accuracy": "Accuracy",
    "predictive accuracy": "Accuracy",
    "test accuracy": "Accuracy",
    "standard accuracy": "Accuracy",
    "accuracy": "Accuracy",

    "precision": "Precision",
    "recall": "Recall",

    "f1 score": "F1",
    "f1 scores": "F1",
    "f1-score": "F1",
    "macro-f1": "F1",
    "f-1 score": "F1",

    "auc": "AUC",
    "roc-auc": "ROC-AUC",

    "word error rate": "WER",

    "mean squared error": "MSE",
    "mean absolute error": "MAE",
    "root mean squared error": "RMSE",


    # Models / Algorithms
    "svm": "Support Vector Machine",
    "support vector machine": "Support Vector Machine",
    "support vector machines": "Support Vector Machine",

    "lstm": "Long Short-Term Memory",
    "lstms": "Long Short-Term Memory",
    "long short-term memory": "Long Short-Term Memory",

    "gru": "Gated Recurrent Unit",
    "gated recurrent unit": "Gated Recurrent Unit",

    "vae": "Variational Autoencoder",
    "vaes": "Variational Autoencoder",
    "variational autoencoder": "Variational Autoencoder",
    "variational autoencoders": "Variational Autoencoder",

    "mlp": "Multi-Layer Perceptron",
    "multi-layer perceptron": "Multi-Layer Perceptron",

    "mcmc": "Markov Chain Monte Carlo",
    "markov chain monte carlo": "Markov Chain Monte Carlo",

    "reinforcement learning": "Reinforcement Learning",
    "deep reinforcement learning": "Deep Reinforcement Learning",

    "gradient descent": "Gradient Descent",

    "gaussian process": "Gaussian Process",
    "gaussian processes": "Gaussian Process",

    "autoencoder": "Autoencoder",
    "autoencoders": "Autoencoder",

    "k-means": "K-Means",
    "kmeans": "K-Means",

    "artificial neural network": "Artificial Neural Network",
    "artificial neural networks": "Artificial Neural Network"
})

In [ ]:
bad_entities.update({
    "reinforcement learning"
})

manual_bad_entities.update({
    "multi-task learning"
})

### 2.4. Create the gold data

In [ ]:
gold = []

for item in tqdm(annotations):

    entities = json.loads(
        item["annotation"]
    )["entities"]

    cleaned = []
    seen = set()

    for ent in entities:

        entity = ent["entity"].strip()
        label = ent["label"]

        # --------------------
        # TASK normalization
        # --------------------
        if label == "TASK":
            entity = entity.lower()

        entity_lower = entity.lower()

        # --------------------
        # Remove tiny entities
        # --------------------
        if len(entity) < 3:
            continue

        # --------------------
        # Remove generic junk
        # --------------------
        if entity_lower in {
            "method",
            "approach",
            "framework",
            "model"
        }:
            continue

        # --------------------
        # Remove bad entities
        # --------------------
        if (
            entity_lower in bad_entities
            or entity_lower in manual_bad_entities
        ):
            continue

        # --------------------
        # Canonical mapping
        # --------------------
        if entity_lower in canonical:
            entity = canonical[entity_lower]

        # --------------------
        # Duplicate removal
        # --------------------
        key = (
            entity.lower(),
            label
        )

        if key in seen:
            continue

        seen.add(key)

        cleaned.append({
            "entity": entity,
            "label": label
        })

    gold.append({
        "id": item["id"],
        "abstract": item["abstract"],
        "entities": cleaned
    })

print("\nGold annotations created :", len(gold))

100%|██████████| 3426/3426 [00:00<00:00, 79378.69it/s]


Gold annotations created : 3426


In [ ]:
from collections import Counter

algo_counter = Counter()
dataset_counter = Counter()
task_counter = Counter()
metric_counter = Counter()

for paper in gold:

    for ent in paper["entities"]:

        if ent["label"] == "ALGORITHM":
            algo_counter[ent["entity"]] += 1

        elif ent["label"] == "DATASET":
            dataset_counter[ent["entity"]] += 1

        elif ent["label"] == "TASK":
            task_counter[ent["entity"]] += 1

        elif ent["label"] == "METRIC":
            metric_counter[ent["entity"]] += 1


print("="*60)
print("ALGORITHM")
print("="*60)

for k, v in algo_counter.most_common(50):
    print(f"{v:3d}  {k}")

print("\n" + "="*60)
print("DATASET")
print("="*60)

for k, v in dataset_counter.most_common(50):
    print(f"{v:3d}  {k}")

print("\n" + "="*60)
print("TASK")
print("="*60)

for k, v in task_counter.most_common(50):
    print(f"{v:3d}  {k}")

print("\n" + "="*60)
print("METRIC")
print("="*60)

for k, v in metric_counter.most_common(50):
    print(f"{v:3d}  {k}")

ALGORITHM
181  Convolutional Neural Network
 76  Generative Adversarial Network
 72  Recurrent Neural Network
 65  Graph Neural Network
 58  Support Vector Machine
 57  Long Short-Term Memory
 53  Transformer
 51  Deep Reinforcement Learning
 51  Stochastic Gradient Descent
 50  Gaussian Process
 49  Deep Neural Network
 47  Variational Autoencoder
 36  Autoencoder
 27  Gradient Descent
 26  logistic regression
 24  Artificial Neural Network
 22  BERT
 22  Markov Chain Monte Carlo
 18  K-Means
 18  Bayesian optimization
 16  ResNet
 16  transfer learning
 15  Random Forest
 14  random forest
 14  PCA
 14  Graph Convolutional Network
 13  ReLU
 12  Q-learning
 11  adversarial training
 10  ADMM
 10  Multi-Layer Perceptron
 10  SVD
  9  knowledge distillation
  9  Gated Recurrent Unit
  9  U-Net
  9  deep convolutional neural network
  9  dynamic programming
  9  ResNet-50
  9  Adam
  9  decision trees
  9  XGBoost
  9  Federated learning
  9  federated learning
  9  NMF
  9  GCNs
  9  v

### 2.5. Save

In [ ]:
with open(
    output_GOLD_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gold,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", output_GOLD_json)

Saved: gold_annotations_3426.json


#### 2.5.1. Preview

In [ ]:
for i in range(5):
    print("\nID:", gold[i]["id"])
    print(gold[i]["entities"])


ID: 0
[{'entity': 'epsilon-Consistent Mixup', 'label': 'ALGORITHM'}, {'entity': 'Mixup', 'label': 'ALGORITHM'}, {'entity': 'SVHN', 'label': 'DATASET'}, {'entity': 'CIFAR-10', 'label': 'DATASET'}, {'entity': 'semi-supervised classification', 'label': 'TASK'}, {'entity': 'Accuracy', 'label': 'METRIC'}]

ID: 1
[{'entity': 'Autoencoder', 'label': 'ALGORITHM'}, {'entity': 'graph-based autoencoders', 'label': 'ALGORITHM'}, {'entity': 'Graph Neural Network', 'label': 'ALGORITHM'}, {'entity': 'anomaly detection', 'label': 'TASK'}]

ID: 2
[{'entity': 'Gaussian models', 'label': 'ALGORITHM'}, {'entity': 'feature selection', 'label': 'TASK'}]

ID: 3
[{'entity': 'Convolutional Neural Network', 'label': 'ALGORITHM'}, {'entity': 'ResNet34', 'label': 'ALGORITHM'}, {'entity': 'ResNet50', 'label': 'ALGORITHM'}, {'entity': 'VGG19', 'label': 'ALGORITHM'}, {'entity': 'DenseNet121', 'label': 'ALGORITHM'}, {'entity': 'skin lesion detection', 'label': 'TASK'}, {'entity': 'Accuracy', 'label': 'METRIC'}]

ID:

### 2.6. Normalize the "gold_annotations_x(i.e no. of rows).json" file

In [ ]:
import json
import re

def normalize_text(text):
    # $...$ -> ...
    text = re.sub(r'\$([^$]+)\$', r'\1', text)

    # \epsilon -> epsilon
    text = re.sub(r'\\([A-Za-z]+)', r'\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Load gold dataset
with open(input_GOLD_json, "r", encoding="utf-8") as f:
    data = json.load(f)

# Normalize abstracts
for item in data:
    if "abstract" in item:
        item["abstract"] = normalize_text(item["abstract"])

# Save new file
with open(
    output_NORMALIZED_GOLD_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Saved:", output_NORMALIZED_GOLD_json)

Saved: normalized_gold_annotations_3426.json


## 🌟 STEP 3 - Perform the BIO tagging and save the "bio_dataset_x(i.e no. of rows).json"

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

# INPUT FILES
input_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"

# OUTPUT FILES
output_BIO_json = f"bio_dataset_{NUM_PAPERS}.json"

### 3.1. Import Libraries &  Load the "normalized_gold_annotations_x(i.e no. of rows).json" file

In [ ]:
import json
import re

Loaded: 1430


In [ ]:
with open(
    input_NORMALIZED_GOLD_json,
    "r",
    encoding="utf-8"
) as f:

    gold = json.load(f)

print("Loaded:", len(gold))

Loaded: 3426


### 3.2. Create the Gold dictionary

In [ ]:
gold_dict = {}

for item in gold:

    gold_dict[
        item["id"]
    ] = item["entities"]

print(
    "Gold dict size:",
    len(gold_dict)
)

Gold dict size: 3426


### 3.3. Generate BIO Dataset

In [ ]:
dataset = []

for paper in gold:

    pid = paper["id"]

    text = paper["abstract"]

    tokens = re.findall(
        r"\w+|[^\w\s]",
        text
    )

    labels = ["O"] * len(tokens)

    entities = gold_dict.get(
        pid,
        []
    )

    entities = sorted(
            entities,
            key=lambda x: len(x["entity"]),
            reverse=True
    )

    lower_tokens = [
        t.lower()
        for t in tokens
    ]

    for ent in entities:

        ent_tokens = re.findall(
            r"\w+|[^\w\s]",
            ent["entity"]
        )

        ent_tokens = [
            x.lower()
            for x in ent_tokens
        ]

        n = len(ent_tokens)

        for i in range(
            len(tokens) - n + 1
        ):

            if (
                lower_tokens[i:i+n]
                ==
                ent_tokens
            ):

                if labels[i] != "O":
                    continue

                labels[i] = (
                    "B-" +
                    ent["label"]
                )

                for j in range(
                    1,
                    n
                ):

                    if (
                        labels[i+j]
                        ==
                        "O"
                    ):

                        labels[i+j] = (
                            "I-" +
                            ent["label"]
                        )

    dataset.append({

        "id": pid,

        "tokens": tokens,

        "ner_tags": labels

    })

print(
    "BIO dataset created:",
    len(dataset)
)

BIO dataset created: 3426


### 3.4. Verify BIO Tags

In [ ]:
for token, tag in zip(

    dataset[0]["tokens"],

    dataset[0]["ner_tags"]

):

    if tag != "O":

        print(
            f"{token:20} {tag}"
        )

epsilon              B-ALGORITHM
-                    I-ALGORITHM
Consistent           I-ALGORITHM
Mixup                I-ALGORITHM
Mixup                B-ALGORITHM
Mixup                B-ALGORITHM
semi                 B-TASK
-                    I-TASK
supervised           I-TASK
classification       I-TASK
accuracy             B-METRIC
SVHN                 B-DATASET
Mixup                B-ALGORITHM
Mixup                B-ALGORITHM


In [ ]:
for token, tag in zip(
    dataset[0]["tokens"],
    dataset[0]["ner_tags"]
):
    if tag != "O":
        print(token, tag)

epsilon B-ALGORITHM
- I-ALGORITHM
Consistent I-ALGORITHM
Mixup I-ALGORITHM
Mixup B-ALGORITHM
Mixup B-ALGORITHM
semi B-TASK
- I-TASK
supervised I-TASK
classification I-TASK
accuracy B-METRIC
SVHN B-DATASET
Mixup B-ALGORITHM
Mixup B-ALGORITHM


In [ ]:
print(gold[0]["entities"])

[{'entity': 'epsilon-Consistent Mixup', 'label': 'ALGORITHM'}, {'entity': 'Mixup', 'label': 'ALGORITHM'}, {'entity': 'SVHN', 'label': 'DATASET'}, {'entity': 'CIFAR-10', 'label': 'DATASET'}, {'entity': 'semi-supervised classification', 'label': 'TASK'}, {'entity': 'Accuracy', 'label': 'METRIC'}]


In [ ]:
print(dataset[0]["tokens"][:50])

['In', 'this', 'paper', 'we', 'propose', 'epsilon', '-', 'Consistent', 'Mixup', '(', 'epsilonmu', ')', '.', 'epsilonmu', 'is', 'a', 'data', '-', 'based', 'structural', 'regularization', 'technique', 'that', 'combines', 'Mixup', "'", 's', 'linear', 'interpolation', 'with', 'consistency', 'regularization', 'in', 'the', 'Mixup', 'direction', ',', 'by', 'compelling', 'a', 'simple', 'adaptive', 'tradeoff', 'between', 'the', 'two', '.', 'This', 'learnable', 'combination']


### 3.5. Save the BIO Dataset Json File

In [ ]:
with open(
    output_BIO_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    output_BIO_json
)

Saved: bio_dataset_3426.json


In [ ]:
matched = 0
total = 0

for paper in gold:

    text = paper["abstract"].lower()

    for ent in paper["entities"]:

        total += 1

        entity = ent["entity"].lower()

        if entity in text:
            matched += 1

print("Total entities :", total)
print("Direct matches :", matched)
print("Coverage % :", round(100 * matched / total, 2))

Total entities : 13032
Direct matches : 12646
Coverage % : 97.04


In [ ]:
assert len(tokens) == len(ner_tags)

NameError: name 'ner_tags' is not defined

## 🌟 STEP 4 - Prep data for Training

In [ ]:
# Part A:
# Loaded the BIO-tagged NER dataset and verified token-label consistency.
# Split the dataset into train (80%), validation (10%), and test (10%) sets.
# Created label mappings (label2id and id2label) and converted BIO tags to numerical IDs.
# Converted the data into Hugging Face Dataset format for model training.

# Part B:
# Loaded the SciBERT tokenizer for scientific text processing.
# Tokenized the input tokens and aligned BIO labels with subword tokens.
# Verified sequence lengths, label IDs, and tokenization outputs.
# Saved the tokenized datasets, making them ready for SciBERT fine-tuning.

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

# input_BIO_json = f"bio_dataset_{NUM_PAPERS}.json"
input_BIO_json = "bio_dataset_3426.json"

label_map_json =  "label_mapping_3426.json"
# f"label_mapping_{NUM_PAPERS}.json"

MODEL_NAME = "allenai/scibert_scivocab_uncased"

### 4.1. Import Libraries &  Load the "bio_dataset_x(i.e no. of rows).json" file

In [ ]:
# !pip install transformers datasets accelerate -q

In [ ]:
import json
import random

from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
with open(input_BIO_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total Papers Loaded: {len(data)}")

Total Papers Loaded: 3426


#### 4.1.1. Check or Confirm if the content/structure of the "bio_dataset_x(i.e no. of rows).json" file is correct

In [ ]:
print(data[0].keys())
print("\n",data[0])

dict_keys(['id', 'tokens', 'ner_tags'])

 {'id': 0, 'tokens': ['In', 'this', 'paper', 'we', 'propose', 'epsilon', '-', 'Consistent', 'Mixup', '(', 'epsilonmu', ')', '.', 'epsilonmu', 'is', 'a', 'data', '-', 'based', 'structural', 'regularization', 'technique', 'that', 'combines', 'Mixup', "'", 's', 'linear', 'interpolation', 'with', 'consistency', 'regularization', 'in', 'the', 'Mixup', 'direction', ',', 'by', 'compelling', 'a', 'simple', 'adaptive', 'tradeoff', 'between', 'the', 'two', '.', 'This', 'learnable', 'combination', 'of', 'consistency', 'and', 'interpolation', 'induces', 'a', 'more', 'flexible', 'structure', 'on', 'the', 'evolution', 'of', 'the', 'response', 'across', 'the', 'feature', 'space', 'and', 'is', 'shown', 'to', 'improve', 'semi', '-', 'supervised', 'classification', 'accuracy', 'on', 'the', 'SVHN', 'and', 'CIFAR10', 'benchmark', 'datasets', ',', 'yielding', 'the', 'largest', 'gains', 'in', 'the', 'most', 'challenging', 'low', 'label', '-', 'availability', 'scenari

In [ ]:
for sample in data:

    assert len(sample["tokens"]) == len(sample["ner_tags"]), (
        f"Mismatch Found in ID: {sample['id']}"
    )

print("All token or tag lengths match.")

All token or tag lengths match.


### 4.2. Create the Train / Validation / Test Split

In [ ]:
train_data, temp_data = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Dataset Split")
print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Dataset Split
Train: 2740
Validation: 343
Test: 343


### 4.3. Build Label Set and Create label2id & id2label and Save

In [ ]:
all_labels = set()

for sample in data:
    all_labels.update(sample["ner_tags"])

all_labels = sorted(list(all_labels))

print("Labels Found:")
print(all_labels)

print("\nTotal Labels:", len(all_labels))

Labels Found:
['B-ALGORITHM', 'B-DATASET', 'B-METRIC', 'B-TASK', 'I-ALGORITHM', 'I-DATASET', 'I-METRIC', 'I-TASK', 'O']

Total Labels: 9


In [ ]:
label2id = {
    label: idx
    for idx, label in enumerate(all_labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print("label2id:")
print(label2id)

print("\nid2label:")
print(id2label)

label2id:
{'B-ALGORITHM': 0, 'B-DATASET': 1, 'B-METRIC': 2, 'B-TASK': 3, 'I-ALGORITHM': 4, 'I-DATASET': 5, 'I-METRIC': 6, 'I-TASK': 7, 'O': 8}

id2label:
{0: 'B-ALGORITHM', 1: 'B-DATASET', 2: 'B-METRIC', 3: 'B-TASK', 4: 'I-ALGORITHM', 5: 'I-DATASET', 6: 'I-METRIC', 7: 'I-TASK', 8: 'O'}


#### 4.3.1. Save Label Mapping

In [ ]:
with open(label_map_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        indent=4
    )

print("Label mapping saved.")

Label mapping saved.


### 4.4. Convert Labels to IDs

In [ ]:
for sample in train_data:

    sample["labels"] = [
        label2id[tag]
        for tag in sample["ner_tags"]
    ]

for sample in val_data:

    sample["labels"] = [
        label2id[tag]
        for tag in sample["ner_tags"]
    ]

for sample in test_data:

    sample["labels"] = [
        label2id[tag]
        for tag in sample["ner_tags"]
    ]

print("Labels converted to IDs.")

Labels converted to IDs.


### 4.5. Convert to Hugging Face Dataset

In [ ]:
train_ds = Dataset.from_list(train_data)

val_ds = Dataset.from_list(val_data)

test_ds = Dataset.from_list(test_data)

print("Hugging Face Dataset created.")

Hugging Face Dataset created.


#### 4.5.1. Verify

In [ ]:
print("Train Dataset")
print(train_ds)

print("\nValidation Dataset")
print(val_ds)

print("\nTest Dataset")
print(test_ds)

Train Dataset
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels'],
    num_rows: 2740
})

Validation Dataset
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels'],
    num_rows: 343
})

Test Dataset
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels'],
    num_rows: 343
})


In [ ]:
print(train_ds[2]["tokens"][:50])
print(train_ds[2]["ner_tags"][:50])
print(train_ds[2]["labels"][:50])

['Reinforcement', 'Learning', '(', 'RL', ')', 'has', 'opened', 'up', 'new', 'opportunities', 'to', 'solve', 'a', 'wide', 'range', 'of', 'complex', 'decision', '-', 'making', 'tasks', '.', 'However', ',', 'modern', 'RL', 'algorithms', ',', 'e', '.', 'g', '.', ',', 'Deep', 'Q', '-', 'Learning', ',', 'are', 'based', 'on', 'deep', 'neural', 'networks', ',', 'putting', 'high', 'computational', 'costs', 'when']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ALGORITHM', 'I-ALGORITHM', 'I-ALGORITHM', 'I-ALGORITHM', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]


In [ ]:
print(train_ds.features)

{'id': Value('int64'), 'tokens': List(Value('string')), 'ner_tags': List(Value('string')), 'labels': List(Value('int64'))}


In [ ]:
from collections import Counter

counter = Counter()

for sample in data:
    counter.update(sample["ner_tags"])

for label, count in sorted(counter.items()):
    print(f"{label:15} {count}")

B-ALGORITHM     10384
B-DATASET       1218
B-METRIC        1430
B-TASK          4074
I-ALGORITHM     14052
I-DATASET       1290
I-METRIC        894
I-TASK          6430
O               648450


### 4.6. Load SciBERT Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("\nTokenizer Loaded")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]


Tokenizer Loaded


In [ ]:
sample = train_ds[0]["tokens"]

print(sample[:20])

encoded = tokenizer(
    sample,
    is_split_into_words=True
)

print(encoded.tokens()[:30])

['Tensor', 'decomposition', 'is', 'a', 'powerful', 'computational', 'tool', 'for', 'multiway', 'data', 'analysis', '.', 'Many', 'popular', 'tensor', 'decomposition', 'approaches', '-', '-', '-']
['[CLS]', 'tensor', 'decomposition', 'is', 'a', 'powerful', 'computational', 'tool', 'for', 'multi', '##way', 'data', 'analysis', '.', 'many', 'popular', 'tensor', 'decomposition', 'approaches', '-', '-', '-', 'such', 'as', 'the', 'tu', '##ck', '##er', 'decomposition', 'and']


### 4.7. Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=512,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):

        word_ids = tokenized_inputs.word_ids(
            batch_index=i
        )

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(
                    label[word_idx]
                )

            else:

                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [ ]:
tokenized_train_ds = train_ds.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_val_ds = val_ds.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_test_ds = test_ds.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/2740 [00:00<?, ? examples/s]

Map:   0%|          | 0/343 [00:00<?, ? examples/s]

Map:   0%|          | 0/343 [00:00<?, ? examples/s]

#### verify

In [ ]:
print(tokenized_train_ds)

print(tokenized_val_ds)

print(tokenized_test_ds)

Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2740
})
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 343
})
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 343
})


In [ ]:
sample = tokenized_train_ds[0]

print(sample.keys())

dict_keys(['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
sample = tokenized_train_ds[0]

tokens = tokenizer.convert_ids_to_tokens(
    sample["input_ids"]
)

for t, l in zip(tokens[:100], sample["labels"][:100]):
    print(t, l)

[CLS] -100
tensor 8
decomposition 8
is 8
a 8
powerful 8
computational 8
tool 8
for 8
multi 8
##way -100
data 8
analysis 8
. 8
many 8
popular 8
tensor 8
decomposition 8
approaches 8
- 8
- 8
- 8
such 8
as 8
the 8
tu 0
##ck -100
##er -100
decomposition 4
and 8
cand 0
##ecom -100
##p -100
/ 4
para 4
##fac -100
( 8
cp 8
) 8
- 8
- 8
- 8
amount 8
to 8
multi 8
- 8
linear 8
factorization 8
. 8
they 8
are 8
insufficient 8
to 8
model 8
( 8
i 8
) 8
complex 8
interactions 8
between 8
data 8
entities 8
, 8
( 8
ii 8
) 8
various 8
data 8
types 8
( 8
e 8
. 8
g 8
. 8
missing 8
data 8
and 8
binary 8
data 8
) 8
, 8
and 8
( 8
iii 8
) 8
noisy 8
observations 8
and 8
outliers 8
. 8
to 8
address 8
these 8
issues 8
, 8
we 8
propose 8
tensor 0
- 4
vari 4


#### Check Sequence Lengths

In [ ]:
lengths = [
    len(x)
    for x in tokenized_train_ds["input_ids"]
]

print("Max Length :", max(lengths))
print("Min Length :", min(lengths))
print("Average Length :", sum(lengths)/len(lengths))

Max Length : 512
Min Length : 30
Average Length : 214.00620437956204


#### Count Truncated Samples

In [ ]:
count = sum(
    1
    for x in tokenized_train_ds["input_ids"]
    if len(x) >= 512
)

print("Sequences reaching 512:", count)
print("Percentage:", round(100 * count / len(tokenized_train_ds), 2), "%")

Sequences reaching 512: 1
Percentage: 0.04 %


#### Verify Label IDs

In [ ]:
valid_ids = set(label2id.values())

invalid = 0

for sample in tokenized_train_ds:

    for lbl in sample["labels"]:

        if lbl == -100:
            continue

        if lbl not in valid_ids:
            invalid += 1

print("Invalid Labels:", invalid)

Invalid Labels: 0


#### Count Labels After Tokenization

In [ ]:
from collections import Counter

counter = Counter()

for sample in tokenized_train_ds:

    for lbl in sample["labels"]:

        if lbl != -100:
            counter[lbl] += 1

for k, v in sorted(counter.items()):
    print(id2label[k], v)

B-ALGORITHM 12887
B-DATASET 1942
B-METRIC 1324
B-TASK 3567
I-ALGORITHM 12279
I-DATASET 1179
I-METRIC 743
I-TASK 5445
O 541531


In [ ]:
print("Train:", len(tokenized_train_ds))
print("Val:", len(tokenized_val_ds))
print("Test:", len(tokenized_test_ds))

print("Labels:", len(label2id))

Train: 2740
Val: 343
Test: 343
Labels: 9


#### Save Tokenized Datasets

In [ ]:
tokenized_train_ds.save_to_disk(
    f"tokenized_train_{NUM_PAPERS}"
)

tokenized_val_ds.save_to_disk(
    f"tokenized_val_{NUM_PAPERS}"
)

tokenized_test_ds.save_to_disk(
    f"tokenized_test_{NUM_PAPERS}"
)

Saving the dataset (0/1 shards):   0%|          | 0/2740 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/343 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/343 [00:00<?, ? examples/s]

## 🌟 STEP 5 - Fine-tuning SciBERT

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

MODEL_NAME = "allenai/scibert_scivocab_uncased"

label_map_json = f"label_mapping_{NUM_PAPERS}.json"

MODEL_DIR = f"scibert_ner_model_trained_on_{NUM_PAPERS}"
MODEL_ZIP = f"scibert_ner_model_trained_on_{NUM_PAPERS}.zip"

### 5.1. Install Libraries

In [ ]:
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import json
import numpy as np

from datasets import load_from_disk

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

### 5.2. Load Tokenized Datasets + Label Mapping + SciBERT Tokenizer + SciBERT Model + Data Collator

In [ ]:
tokenized_train_ds = load_from_disk(
    f"tokenized_train_{NUM_PAPERS}"
)

tokenized_val_ds = load_from_disk(
    f"tokenized_val_{NUM_PAPERS}"
)

tokenized_test_ds = load_from_disk(
    f"tokenized_test_{NUM_PAPERS}"
)

print(tokenized_train_ds)
print(tokenized_val_ds)
print(tokenized_test_ds)

Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2740
})
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 343
})
Dataset({
    features: ['id', 'tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 343
})


In [ ]:
with open(label_map_json, "r") as f:
    mappings = json.load(f)

label2id = mappings["label2id"]

id2label = {
    int(k): v
    for k, v in mappings["id2label"].items()
}

print(label2id)

{'B-ALGORITHM': 0, 'B-DATASET': 1, 'B-METRIC': 2, 'B-TASK': 3, 'I-ALGORITHM': 4, 'I-DATASET': 5, 'I-METRIC': 6, 'I-TASK': 7, 'O': 8}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print("SciBERT Loaded")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  442MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

SciBERT Loaded


In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

### 5.3. Compute Metrics Function

In [ ]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = [
        [
            id2label[p]
            for p, l in zip(pred, lab)
            if l != -100
        ]
        for pred, lab in zip(
            predictions,
            labels
        )
    ]

    true_labels = [
        [
            id2label[l]
            for p, l in zip(pred, lab)
            if l != -100
        ]
        for pred, lab in zip(
            predictions,
            labels
        )
    ]

    return {
        "precision":
            precision_score(
                true_labels,
                true_predictions
            ),
        "recall":
            recall_score(
                true_labels,
                true_predictions
            ),
        "f1":
            f1_score(
                true_labels,
                true_predictions
            ),
        "accuracy":
            accuracy_score(
                true_labels,
                true_predictions
            )
    }

### 5.4. Training Arguments

In [ ]:
training_args = TrainingArguments(

    output_dir=MODEL_DIR,

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    save_total_limit=2,

    logging_steps=50,

    report_to="none",

    seed=SEED
)

In [ ]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

### 5.5. Create Trainer &  Train Model

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train_ds,

    eval_dataset=tokenized_val_ds,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[early_stopping]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.090974,0.085295,0.567859,0.684584,0.620782,0.967792
2,0.059355,0.082905,0.622969,0.679268,0.649901,0.970252
3,0.041820,0.088586,0.643056,0.681040,0.661503,0.971663
4,0.025559,0.100170,0.616709,0.719433,0.664122,0.969862
5,0.022503,0.104972,0.646095,0.698760,0.671396,0.971198


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1715, training_loss=0.05551742831402548, metrics={'train_runtime': 1011.4912, 'train_samples_per_second': 13.544, 'train_steps_per_second': 1.696, 'total_flos': 2199482155847520.0, 'train_loss': 0.05551742831402548, 'epoch': 5.0})

### 5.6. Evaluate on Validation Set + Test Set

In [ ]:
val_results = trainer.evaluate(
    tokenized_val_ds
)

print(val_results)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.022503,0.104972,5,0.646095,0.698760,0.671396,0.971198


{'eval_loss': 0.10497236996889114, 'eval_precision': 0.6460950300382304, 'eval_recall': 0.6987595983461311, 'eval_f1': 0.6713961407491486, 'eval_accuracy': 0.9711975517919023}


In [ ]:
test_results = trainer.evaluate(
    tokenized_test_ds
)

print(test_results)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.022503,0.086613,5,0.657205,0.712005,0.683508,0.974967


{'eval_loss': 0.08661332726478577, 'eval_precision': 0.6572052401746725, 'eval_recall': 0.7120047309284447, 'eval_f1': 0.683508373545274, 'eval_accuracy': 0.9749666221628839}


### 5.7. Save Final Model

In [ ]:
MODEL_DIR

'scibert_ner_model_trained_on_3426'

In [ ]:
trainer.save_model(MODEL_DIR)

tokenizer.save_pretrained(MODEL_DIR)

print("Model Saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved


In [ ]:
import shutil

shutil.make_archive(
    MODEL_DIR,
    "zip",
    MODEL_DIR
)

print("ZIP Created")

ZIP Created


In [ ]:
print(MODEL_DIR)

scibert_ner_model_trained_on_3426


## 🌟 STEP 6 - Load & Evaluate the trained NER model

### → Change values here

In [ ]:
# ==========================
# CONFIGURATION
# ==========================

input_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"

model_path_check = "/content/drive/MyDrive/Colab Notebooks/CBSOT Internship/Project-2 Files" ##

MODEL_NAME = f"scibert_ner_model_trained_on_{NUM_PAPERS}"

model_path = f"/content/drive/MyDrive/Colab Notebooks/CBSOT Internship/Project-2 Files/{MODEL_NAME}"

OUTPUT_pred_JSON = f"prediction_on_{NUM_PAPERS}.json"

### 6.1. Import Lib & Load "normalized_gold_annotations_x.json" file

In [ ]:
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

In [ ]:
input_NORMALIZED_GOLD_json = f"normalized_gold_annotations_{NUM_PAPERS}.json"

with open(input_NORMALIZED_GOLD_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total papers:", len(data))

Total papers: 3426


#### 6.1.1. Mount the Drive and Load Trained SciBERT NER Model

In [ ]:
import os
from google.colab import drive

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")
else:
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")

Google Drive is already mounted.


In [ ]:
import os
model_path_check = "/content/drive/MyDrive/Colab Notebooks/CBSOT Internship/Project-2 Files" ##
print("Files/Folders in 'Project-2 Files':")
print(os.listdir(model_path_check))
print()
print(f"Files inside '{MODEL_NAME}':")
print(os.listdir(model_path))

Files/Folders in 'Project-2 Files':
['paper_embeddings.npy', 'paper_faiss.index', 'processed_df.pkl', 'saved_scibert_ner', 'scibert_ner_model_trained_on_3426']

Files inside 'scibert_ner_model_trained_on_3426':
['model.safetensors', 'training_args.bin', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

### 6.2. Define the "extract_entities()" function

In [ ]:
def extract_entities(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(
        outputs.logits,
        dim=2
    )

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    labels = [
        model.config.id2label[p.item()]
        for p in predictions[0]
    ]

    results = []

    current_tokens = []
    current_label = None

    # Convert BIO labels to entities
    for token, label in zip(tokens, labels):

        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        if label.startswith("B-"):

            if current_tokens:
                results.append({
                    "entity": tokenizer.convert_tokens_to_string(
                        current_tokens
                    ).lower().strip(),
                    "label": current_label
                })

            current_tokens = [token]
            current_label = label[2:]

        elif (
            label.startswith("I-")
            and current_label == label[2:]
        ):

            current_tokens.append(token)

        else:

            if current_tokens:
                results.append({
                    "entity": tokenizer.convert_tokens_to_string(
                        current_tokens
                    ).lower().strip(),
                    "label": current_label
                })

            current_tokens = []
            current_label = None

    # Save last entity
    if current_tokens:
        results.append({
            "entity": tokenizer.convert_tokens_to_string(
                current_tokens
            ).lower().strip(),
            "label": current_label
        })

    return results

#### 6.2.1. Check

In [ ]:
paper = data[0]

print("GOLD")
print(paper["entities"])

print("\nPRED")
print(extract_entities(paper["abstract"]))

GOLD
[{'entity': 'epsilon-Consistent Mixup', 'label': 'ALGORITHM'}, {'entity': 'Mixup', 'label': 'ALGORITHM'}, {'entity': 'SVHN', 'label': 'DATASET'}, {'entity': 'CIFAR-10', 'label': 'DATASET'}, {'entity': 'semi-supervised classification', 'label': 'TASK'}, {'entity': 'Accuracy', 'label': 'METRIC'}]

PRED
[{'entity': 'epsilon - consistent mixup', 'label': 'ALGORITHM'}, {'entity': 'epsilonmu', 'label': 'ALGORITHM'}, {'entity': 'epsilonmu', 'label': 'ALGORITHM'}, {'entity': 'mixup', 'label': 'ALGORITHM'}, {'entity': 'semi - supervised classification', 'label': 'TASK'}, {'entity': 'accuracy', 'label': 'METRIC'}, {'entity': 'svhn', 'label': 'DATASET'}, {'entity': 'cifar10', 'label': 'DATASET'}, {'entity': 'epsilonmu', 'label': 'ALGORITHM'}, {'entity': 'mixup', 'label': 'ALGORITHM'}, {'entity': 'epsilonmu', 'label': 'ALGORITHM'}, {'entity': 'epsilonmu', 'label': 'ALGORITHM'}, {'entity': 'mixup', 'label': 'ALGORITHM'}]


In [ ]:
inputs = tokenizer(
    paper["abstract"],
    return_tensors="pt",
    truncation=True,
    max_length=512
)

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

labels = [
    model.config.id2label[p.item()]
    for p in predictions[0]
]

for token, label in zip(tokens, labels):

    if token in ["[CLS]", "[SEP]", "[PAD]"]:
        continue

    if "eps" in token or "##il" in token or "##on" in token:
        print(token, " ---> ", label)

eps  --->  B-ALGORITHM
##il  --->  I-ALGORITHM
##on  --->  I-ALGORITHM
eps  --->  B-ALGORITHM
##il  --->  I-ALGORITHM
##on  --->  I-ALGORITHM
eps  --->  B-ALGORITHM
##il  --->  I-ALGORITHM
##on  --->  I-ALGORITHM
eps  --->  B-ALGORITHM
##il  --->  I-ALGORITHM
##on  --->  I-ALGORITHM
eps  --->  B-ALGORITHM
##il  --->  I-ALGORITHM
##on  --->  I-ALGORITHM
eps  --->  B-ALGORITHM
##il  --->  I-ALGORITHM
##on  --->  I-ALGORITHM


#### 6.2.2. Normalize

In [ ]:
import re

def normalize_entity(text):

    text = text.lower().strip()

    # remove spaces
    text = re.sub(r"\s+", "", text)

    # remove hyphens
    text = text.replace("-", "")

    return text

#### 6.2.3. Predict the entities for first 10 papers

In [ ]:
correct = 0
gold_total = 0
pred_total = 0

sample_data = data[:10]

for i, paper in enumerate(sample_data):

    gold = {
        (
            normalize_entity(e["entity"]),
            e["label"]
        )
        for e in paper["entities"]
    }

    preds = extract_entities(paper["abstract"])

    pred = {
        (
            normalize_entity(e["entity"]),
            e["label"]
        )
        for e in preds
    }

    # Evaluation
    correct += len(gold & pred)
    gold_total += len(gold)
    pred_total += len(pred)

    # Per-paper analysis
    print("\n" + "=" * 100)
    print(f"PAPER #{i+1}")

    print("\nMATCHES")
    print(gold & pred)

    print("\nMISSED (Gold but not Pred)")
    print(gold - pred)

    print("\nEXTRA (Pred but not Gold)")
    print(pred - gold)


# ==========================
# OVERALL METRICS
# ==========================

precision = correct / pred_total if pred_total else 0
recall = correct / gold_total if gold_total else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0
)

print("\n" + "=" * 100)
print("FINAL RESULTS")
print("=" * 100)

print("Gold      :", gold_total)
print("Predicted :", pred_total)
print("Correct   :", correct)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")


PAPER #1

MATCHES
{('accuracy', 'METRIC'), ('epsilonconsistentmixup', 'ALGORITHM'), ('cifar10', 'DATASET'), ('semisupervisedclassification', 'TASK'), ('svhn', 'DATASET'), ('mixup', 'ALGORITHM')}

MISSED (Gold but not Pred)
set()

EXTRA (Pred but not Gold)
{('epsilonmu', 'ALGORITHM')}

PAPER #2

MATCHES
{('anomalydetection', 'TASK'), ('graphneuralnetwork', 'ALGORITHM'), ('graphbasedautoencoders', 'ALGORITHM')}

MISSED (Gold but not Pred)
{('autoencoder', 'ALGORITHM')}

EXTRA (Pred but not Gold)
set()

PAPER #3

MATCHES
{('gaussianmodels', 'ALGORITHM'), ('featureselection', 'TASK')}

MISSED (Gold but not Pred)
set()

EXTRA (Pred but not Gold)
{('jointly', 'ALGORITHM')}

PAPER #4

MATCHES
{('convolutionalneuralnetwork', 'ALGORITHM'), ('accuracy', 'METRIC'), ('skinlesiondetection', 'TASK')}

MISSED (Gold but not Pred)
{('resnet50', 'ALGORITHM'), ('vgg19', 'ALGORITHM'), ('densenet121', 'ALGORITHM'), ('resnet34', 'ALGORITHM')}

EXTRA (Pred but not Gold)
{('resnet', 'ALGORITHM'), ('vgg', 'AL

### 6.3. Predict & Save "prediction_on_x(i.e no. of rows).json"

In [ ]:
import json

correct = 0
gold_total = 0
pred_total = 0

all_predictions = []

SAVE_EVERY = 20

for idx, paper in enumerate(data, start=1):

    abstract = paper["abstract"]

    gold = {
        (normalize_entity(e["entity"]), e["label"])
        for e in paper["entities"]
    }

    preds = extract_entities(abstract)

    # Save predictions
    all_predictions.append({
            "id": paper["id"],
            "abstract": abstract,
            "predicted_entities": preds
    })

    pred = {
        (normalize_entity(e["entity"]), e["label"])
        for e in preds
    }

    correct += len(gold & pred)
    gold_total += len(gold)
    pred_total += len(pred)

    # Save every 20 papers
    if idx % SAVE_EVERY == 0:
        with open(OUTPUT_pred_JSON, "w", encoding="utf-8") as f:
            json.dump(all_predictions, f, indent=2, ensure_ascii=False)

        print(f"Saved progress: {idx} papers")

# Final save
with open(OUTPUT_pred_JSON, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, indent=2, ensure_ascii=False)

print("\nFinal save completed.")

Saved progress: 20 papers
Saved progress: 40 papers
Saved progress: 60 papers
Saved progress: 80 papers
Saved progress: 100 papers
Saved progress: 120 papers
Saved progress: 140 papers
Saved progress: 160 papers
Saved progress: 180 papers
Saved progress: 200 papers
Saved progress: 220 papers
Saved progress: 240 papers
Saved progress: 260 papers
Saved progress: 280 papers
Saved progress: 300 papers
Saved progress: 320 papers
Saved progress: 340 papers
Saved progress: 360 papers
Saved progress: 380 papers
Saved progress: 400 papers
Saved progress: 420 papers
Saved progress: 440 papers
Saved progress: 460 papers
Saved progress: 480 papers
Saved progress: 500 papers
Saved progress: 520 papers
Saved progress: 540 papers
Saved progress: 560 papers
Saved progress: 580 papers
Saved progress: 600 papers
Saved progress: 620 papers
Saved progress: 640 papers
Saved progress: 660 papers
Saved progress: 680 papers
Saved progress: 700 papers
Saved progress: 720 papers
Saved progress: 740 papers
Saved

### 6.4. Compute Precision, Recall, F1 & Display Results

In [ ]:
precision = (
    correct / pred_total
    if pred_total > 0
    else 0
)

recall = (
    correct / gold_total
    if gold_total > 0
    else 0
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

In [ ]:
print("FINAL RESULTS")

print("Gold Entities      :", gold_total)
print("Predicted Entities :", pred_total)
print("Correct Matches    :", correct)

print(f"\nPrecision = {precision:.4f}")
print(f"Recall    = {recall:.4f}")
print(f"F1        = {f1:.4f}")

FINAL RESULTS
Gold Entities      : 13022
Predicted Entities : 13008
Correct Matches    : 10797

Precision = 0.8300
Recall    = 0.8291
F1        = 0.8296


In [ ]:
print(all_predictions[0])

{'id': 0, 'abstract': "In this paper we propose epsilon-Consistent Mixup (epsilonmu). epsilonmu is a data-based structural regularization technique that combines Mixup's linear interpolation with consistency regularization in the Mixup direction, by compelling a simple adaptive tradeoff between the two. This learnable combination of consistency and interpolation induces a more flexible structure on the evolution of the response across the feature space and is shown to improve semi-supervised classification accuracy on the SVHN and CIFAR10 benchmark datasets, yielding the largest gains in the most challenging low label-availability scenarios. Empirical studies comparing epsilonmu and Mixup are presented and provide insight into the mechanisms behind epsilonmu's effectiveness. In particular, epsilonmu is found to produce more accurate synthetic labels and more confident predictions than Mixup.", 'predicted_entities': [{'entity': 'epsilon - consistent mixup', 'label': 'ALGORITHM'}, {'enti

### 6.5. Save only the unique entities in "prediction_unique_entities_on_x(i.e no. of rows).json"

In [ ]:
OUTPUT_unique_pred_JSON = f"predicted_unique_entities_on_{NUM_PAPERS}.json"

In [ ]:
unique_predictions = []

for paper in all_predictions:

    seen = set()
    unique_entities = []

    for ent in paper["predicted_entities"]:

        key = (
            ent["entity"].lower().strip(),
            ent["label"]
        )

        if key not in seen:

            seen.add(key)

            unique_entities.append({
                "entity": ent["entity"],
                "label": ent["label"]
            })

    unique_predictions.append({
        "id": paper["id"],
        "abstract": paper["abstract"],
        "predicted_entities": unique_entities
    })

In [ ]:
with open(
    OUTPUT_unique_pred_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unique_predictions,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", OUTPUT_unique_pred_JSON)

Saved: predicted_entities_unique_3426.json
